# Exploratory Data Analysis with SQL
Uses SQLite and the IBM course SQL dataset. The notebook prints the schema first and derives a binary success flag for yearly rates.

In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
df = pd.read_csv(DATA_URL)
df.columns = [c.strip() for c in df.columns]
print("Columns:", df.columns.tolist())
display(df.head())
conn = sqlite3.connect(":memory:")
df.to_sql("SPACEXTBL", conn, index=False, if_exists="replace")

def run_query(title, query):
    print("\\n" + title + "\\n" + query)
    result = pd.read_sql_query(query, conn)
    display(result)
    return result

In [ ]:
run_query("1. Unique launch sites", "SELECT DISTINCT Launch_Site FROM SPACEXTBL ORDER BY Launch_Site;")
run_query("2. First five CCA launch-site records", "SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5;")
run_query("3. Total payload mass", "SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_mass_kg FROM SPACEXTBL;")
run_query("4. Average payload mass for F9 v1.1", "SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_mass_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%';")
run_query("5. Landing outcomes count", "SELECT Landing_Outcome, COUNT(*) AS total FROM SPACEXTBL GROUP BY Landing_Outcome ORDER BY total DESC;")
run_query("6. Payload mass by launch site", "SELECT Launch_Site, SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL GROUP BY Launch_Site ORDER BY total_payload_kg DESC;")

In [ ]:
run_query("7. Yearly launch counts", "SELECT strftime('%Y', Date) AS launch_year, COUNT(*) AS launches FROM SPACEXTBL GROUP BY launch_year ORDER BY launch_year;")
yearly_success_query = "SELECT strftime('%Y', Date) AS launch_year, ROUND(100.0 * AVG(CASE WHEN Landing_Outcome LIKE 'Success%' THEN 1.0 ELSE 0.0 END), 2) AS success_rate_percent FROM SPACEXTBL GROUP BY launch_year ORDER BY launch_year;"
run_query("8. Yearly landing success rate", yearly_success_query)
run_query("9. Successful drone-ship landings with payload 4000–6000 kg", "SELECT Booster_Version, PAYLOAD_MASS__KG_, Landing_Outcome FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;")
run_query("10. Maximum payload records", "SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);")
run_query("11. 2015 landing failures", "SELECT Date, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Date LIKE '2015%' AND Landing_Outcome LIKE '%Failure%';")


## Notes
Run the queries and use the displayed outputs in your report. If the hosted course CSV schema changes, inspect the printed column list and adjust SQL field names accordingly. Do not paste guessed counts or success percentages.